In [1]:
import sys; sys.path.append("..")
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from src.data import load_data
from src.config import RANDOM_STATE, CV_DIR
from src.experiment_log import ExperimentLogger

X_train, _, y_train, _, labels = load_data()
lg = ExperimentLogger("D")

param_dist = {
    "n_estimators": [100, 150, 250],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.05, 0.1, 0.2],
    "subsample": [0.7, 0.85, 1.0],
    "colsample_bytree": [0.7, 0.85, 1.0],
    "min_child_weight": [0.1, 1, 3],
}
base = XGBClassifier(tree_method="hist", objective="multi:softprob",
                     n_jobs=-1, random_state=RANDOM_STATE)
cv3 = StratifiedKFold(3, shuffle=True, random_state=RANDOM_STATE)

rs = RandomizedSearchCV(base, param_dist, n_iter=10, scoring="f1_macro", cv=cv3,
                        random_state=RANDOM_STATE, n_jobs=1, verbose=2)
rs.fit(X_train, y_train)
pd.DataFrame(rs.cv_results_).to_csv(CV_DIR / "tuning_xgb_random.csv", index=False)
print(rs.best_params_, round(rs.best_score_, 4))

best = XGBClassifier(**rs.best_params_, tree_method="hist", objective="multi:softprob",
                     n_jobs=-1, random_state=RANDOM_STATE)
lg.run(best, X_train, y_train, "xgb_tuned",
       notes=f"RandomizedSearch 3-fold n_iter=10 cv={rs.best_score_:.4f}, bo sung sau khi khoa mo hinh cuoi")

Fitting 3 folds for each of 10 candidates, totalling 30 fits
[CV] END colsample_bytree=0.85, learning_rate=0.2, max_depth=8, min_child_weight=1, n_estimators=100, subsample=0.7; total time=  12.9s
[CV] END colsample_bytree=0.85, learning_rate=0.2, max_depth=8, min_child_weight=1, n_estimators=100, subsample=0.7; total time=  10.7s
[CV] END colsample_bytree=0.85, learning_rate=0.2, max_depth=8, min_child_weight=1, n_estimators=100, subsample=0.7; total time=  11.1s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=8, min_child_weight=1, n_estimators=150, subsample=0.85; total time=  16.3s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=8, min_child_weight=1, n_estimators=150, subsample=0.85; total time=  16.4s
[CV] END colsample_bytree=0.7, learning_rate=0.1, max_depth=8, min_child_weight=1, n_estimators=150, subsample=0.85; total time=  16.2s
[CV] END colsample_bytree=0.85, learning_rate=0.05, max_depth=8, min_child_weight=0.1, n_estimators=150, subsample=1.0; tot

,accuracy,balanced_acc,macro_f1,weighted_f1,mcc,recall_c0,recall_c1,recall_c2,recall_c3,recall_c4,...,recall_c6,recall_c7,recall_c8,recall_c9,recall_c10,recall_c11,model,fold,fit_time,pred_time
0,0.998251,0.958975,0.975673,0.998245,0.995687,0.990984,0.964706,1.0,0.998489,0.833333,...,1.0,1.000000,0.983051,1.000000,0.994286,0.942857,xgb_tuned,0,18.134122,0.195333
1,0.997721,0.931895,0.947965,0.997699,0.994382,0.991803,0.988235,1.0,1.000000,0.500000,...,1.0,0.993789,0.983051,0.996885,0.986122,0.942857,xgb_tuned,1,17.483267,0.167678
2,0.998039,0.958539,0.951321,0.998048,0.995165,0.993443,1.000000,1.0,1.000000,0.800000,...,1.0,0.993789,0.983051,0.996885,0.985306,1.000000,xgb_tuned,2,17.715232,0.195926
3,0.998357,0.981004,0.981353,0.998356,0.995948,0.992623,0.976471,1.0,0.995475,0.833333,...,1.0,1.000000,0.980676,1.000000,0.993464,1.000000,xgb_tuned,3,17.529407,0.190512
4,0.998251,0.951930,0.971467,0.998241,0.995686,0.994262,0.952941,1.0,0.998492,0.833333,...,1.0,1.000000,0.992754,0.996894,0.990196,0.914286,xgb_tuned,4,17.574647,0.250189
